# Módulo 8 · Clase 10 — Proyecto: Clasificador de Imágenes con TensorFlow

**Curso de Inteligencia Artificial — Nivel básico**

---

## Objetivo del proyecto

Construir un **clasificador de imágenes profesional** para CIFAR-10 (10 categorías: aviones, automóviles, pájaros, gatos, ciervos, perros, ranas, caballos, barcos, camiones) aplicando todos los conceptos del módulo:

| Fase | Contenido |
|------|-----------|
| 1 | Exploración y preprocesado del dataset |
| 2 | Baseline: red densa |
| 3 | CNN desde cero con regularización |
| 4 | Transfer Learning con MobileNetV2 |
| 5 | Comparativa y diagnóstico final |
| 6 | Guardar, cargar y usar el modelo en producción |

---

```
CIFAR-10:
  60,000 imágenes en color (RGB) de 32×32 píxeles
  10 clases balanceadas, 6,000 imágenes por clase
  50,000 train + 10,000 test
  Referencia: LeCun & Krizhevsky (2009)
```

---
## Fase 1 — Exploración y preprocesado

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.metrics import classification_report, confusion_matrix
import os, datetime, json

tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU: {len(gpus)} disponibles {'→ entrenamiento acelerado' if gpus else '(CPU mode)'}")

NOMBRES_CLASES = ['avión','auto','pájaro','gato','ciervo','perro','rana','caballo','barco','camión']

In [ ]:
# Cargar CIFAR-10
(X_tr_raw, y_tr_raw), (X_te_raw, y_te_raw) = keras.datasets.cifar10.load_data()
y_tr = y_tr_raw.ravel()
y_te = y_te_raw.ravel()

print(f"Train: {X_tr_raw.shape} → {y_tr.shape}")
print(f"Test:  {X_te_raw.shape} → {y_te.shape}")
print(f"Dtype: {X_tr_raw.dtype}, rango: [{X_tr_raw.min()}, {X_tr_raw.max()}]")

# Distribución de clases
conteos = np.bincount(y_tr)
print(f"\nClases (train):")
for i, (nombre, n) in enumerate(zip(NOMBRES_CLASES, conteos)):
    print(f"  {i}: {nombre:10} → {n:,} imágenes")

In [ ]:
# Visualizar muestras del dataset
fig, axes = plt.subplots(10, 8, figsize=(16, 20))
for clase in range(10):
    idx_clase = np.where(y_tr == clase)[0][:8]
    for col, idx in enumerate(idx_clase):
        axes[clase, col].imshow(X_tr_raw[idx])
        if col == 0:
            axes[clase, col].set_ylabel(NOMBRES_CLASES[clase], fontsize=11, rotation=0,
                                         labelpad=55, va='center')
        axes[clase, col].axis('off')
plt.suptitle('CIFAR-10: 8 muestras por clase', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Preprocesado
# Normalizar a [0, 1]
X_tr_norm = X_tr_raw.astype('float32') / 255.0
X_te_norm = X_te_raw.astype('float32') / 255.0

# Para modelos densos: aplanar
X_tr_flat = X_tr_norm.reshape(-1, 32*32*3)
X_te_flat = X_te_norm.reshape(-1, 32*32*3)

# Estadísticas
mean = X_tr_norm.mean(axis=(0,1,2))
std  = X_tr_norm.std(axis=(0,1,2))
print(f"Media por canal RGB: {mean.round(4)}")
print(f"Std  por canal RGB:  {std.round(4)}")

# Normalización por canal (para modelos más avanzados)
X_tr_std = (X_tr_norm - mean) / std
X_te_std  = (X_te_norm - mean) / std

print(f"\nDespués de normalizar por canal:")
print(f"  Media: {X_tr_std.mean(axis=(0,1,2)).round(4)}")
print(f"  Std:   {X_tr_std.std(axis=(0,1,2)).round(4)}")

---
## Fase 2 — Baseline: red densa

In [ ]:
# Modelo baseline: red densa simple
# Propósito: fijar el nivel mínimo que debe superar la CNN
print("=" * 55)
print("FASE 2: BASELINE — RED DENSA")
print("=" * 55)

tf.random.set_seed(42)
modelo_dense = keras.Sequential([
    layers.Input(shape=(3072,)),   # 32×32×3
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax'),
], name='baseline_dense')

modelo_dense.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
modelo_dense.summary()

hist_dense = modelo_dense.fit(
    X_tr_flat, y_tr,
    epochs=20, batch_size=256,
    validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping('val_loss', patience=5, restore_best_weights=True)],
    verbose=1
)

_, acc_dense = modelo_dense.evaluate(X_te_flat, y_te, verbose=0)
print(f"\nBaseline Dense → Test accuracy: {acc_dense:.4f} ({acc_dense:.2%})")

---
## Fase 3 — CNN desde cero con regularización

In [ ]:
print("=" * 55)
print("FASE 3: CNN DESDE CERO")
print("=" * 55)

# Data augmentation (solo se aplica durante el entrenamiento)
augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
], name='augmentation')

tf.random.set_seed(42)

entradas = keras.Input(shape=(32, 32, 3))
x = augmentation(entradas)

# Bloque 1: 32×32 → 16×16
x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.2)(x)

# Bloque 2: 16×16 → 8×8
x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.3)(x)

# Bloque 3: 8×8 → 4×4
x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.4)(x)

# Clasificador
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.4)(x)
salidas = layers.Dense(10, activation='softmax')(x)

modelo_cnn = keras.Model(entradas, salidas, name='CNN_cifar10')
modelo_cnn.summary()

In [ ]:
# Compilar con LR schedule
N_EPOCAS   = 60
BATCH_SIZE = 128
pasos_ep   = len(X_tr_norm) * 0.9 // BATCH_SIZE

lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.001,
    decay_steps=int(N_EPOCAS * pasos_ep),
    alpha=1e-5
)

modelo_cnn.compile(
    optimizer=keras.optimizers.Adam(lr_schedule),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

log_dir_cnn = 'logs/proyecto/cnn/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
callbacks_cnn = [
    keras.callbacks.EarlyStopping('val_accuracy', patience=15, restore_best_weights=True, mode='max'),
    keras.callbacks.ModelCheckpoint('cifar10_cnn_best.keras', monitor='val_accuracy',
                                     save_best_only=True, mode='max', verbose=0),
    keras.callbacks.TensorBoard(log_dir_cnn, histogram_freq=0),
]

hist_cnn = modelo_cnn.fit(
    X_tr_norm, y_tr,
    epochs=N_EPOCAS, batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=callbacks_cnn,
    verbose=1
)

_, acc_cnn = modelo_cnn.evaluate(X_te_norm, y_te, verbose=0)
n_ep_cnn   = len(hist_cnn.history['loss'])
print(f"\nCNN desde cero ({n_ep_cnn} épocas) → Test accuracy: {acc_cnn:.4f} ({acc_cnn:.2%})")

In [ ]:
def plot_historial(hist, titulo):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist.history['loss'],     'steelblue', lw=2, label='Train')
    axes[0].plot(hist.history['val_loss'], 'coral',     lw=2, label='Val')
    axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{titulo} — Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(hist.history['accuracy'],     'steelblue', lw=2, label='Train')
    axes[1].plot(hist.history['val_accuracy'], 'coral',     lw=2, label='Val')
    axes[1].set_xlabel('Época'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title(f'{titulo} — Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

plot_historial(hist_cnn, 'CNN desde cero')

---
## Fase 4 — Transfer Learning con MobileNetV2

In [ ]:
print("=" * 55)
print("FASE 4: TRANSFER LEARNING — MobileNetV2")
print("=" * 55)

# CIFAR-10 es 32×32, MobileNetV2 necesita ≥96×96 → resize
TARGET_SIZE = 96

print(f"Redimensionando imágenes a {TARGET_SIZE}×{TARGET_SIZE}...")
X_tr_up = tf.image.resize(X_tr_norm, (TARGET_SIZE, TARGET_SIZE)).numpy()
X_te_up = tf.image.resize(X_te_norm, (TARGET_SIZE, TARGET_SIZE)).numpy()
print(f"Train: {X_tr_up.shape}")

# Base preentrenada
base_model = keras.applications.MobileNetV2(
    input_shape=(TARGET_SIZE, TARGET_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False
print(f"MobileNetV2 base: {base_model.count_params():,} parámetros (congelados)")

# Augmentation para TL
augm_tl = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# Modelo TL
tf.random.set_seed(42)
entradas_tl = keras.Input(shape=(TARGET_SIZE, TARGET_SIZE, 3))
x = augm_tl(entradas_tl)
x = keras.applications.mobilenet_v2.preprocess_input(x * 255)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
salidas_tl = layers.Dense(10, activation='softmax')(x)

modelo_tl = keras.Model(entradas_tl, salidas_tl, name='MobileNetV2_TL')

params_trainable = sum(v.numpy().size for v in modelo_tl.trainable_variables)
print(f"Parámetros entrenables: {params_trainable:,} (solo el clasificador)")

# Fase 4a: entrenar solo el clasificador
modelo_tl.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nFase 4a: entrenando clasificador (base congelada)...")
hist_tl1 = modelo_tl.fit(
    X_tr_up, y_tr,
    epochs=10, batch_size=128,
    validation_split=0.1, verbose=1
)

_, acc_tl1 = modelo_tl.evaluate(X_te_up, y_te, verbose=0)
print(f"\nTL fase 1 → Test accuracy: {acc_tl1:.4f}")

In [ ]:
# Fase 4b: fine-tuning — descongelar últimas capas
print("Fase 4b: fine-tuning (últimas 30 capas de MobileNetV2)...")

base_model.trainable = True
for capa in base_model.layers[:-30]:
    capa.trainable = False

params_ft = sum(v.numpy().size for v in modelo_tl.trainable_variables)
print(f"Parámetros entrenables tras fine-tuning: {params_ft:,}")

# lr muy baja para no destruir los pesos preentrenados
modelo_tl.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

hist_tl2 = modelo_tl.fit(
    X_tr_up, y_tr,
    epochs=15, batch_size=64,
    validation_split=0.1,
    callbacks=[
        keras.callbacks.EarlyStopping('val_accuracy', patience=8, restore_best_weights=True, mode='max'),
        keras.callbacks.ModelCheckpoint('cifar10_tl_best.keras', monitor='val_accuracy',
                                         save_best_only=True, mode='max', verbose=0),
    ],
    verbose=1
)

_, acc_tl2 = modelo_tl.evaluate(X_te_up, y_te, verbose=0)
print(f"\nTL + fine-tuning → Test accuracy: {acc_tl2:.4f}")

---
## Fase 5 — Comparativa y diagnóstico final

In [ ]:
print("=" * 65)
print("FASE 5: COMPARATIVA FINAL")
print("=" * 65)

print(f"\n{'Modelo':35} {'Params':>12} {'Test acc':>10}")
print('-' * 60)
for nombre, m, acc in [
    ('Baseline Dense',           modelo_dense, acc_dense),
    ('CNN desde cero',           modelo_cnn,   acc_cnn),
    ('TL: MobileNetV2 (solo clf)',modelo_tl,   acc_tl1),
    ('TL: MobileNetV2 + FT',     modelo_tl,   acc_tl2),
]:
    p = m.count_params()
    print(f"{nombre:35} {p:>12,} {acc:>10.4f}")

In [ ]:
# Análisis de errores del mejor modelo
mejor_modelo = modelo_tl if acc_tl2 > acc_cnn else modelo_cnn
X_te_mejor   = X_te_up  if acc_tl2 > acc_cnn else X_te_norm
nombre_mejor = 'TL + Fine-tuning' if acc_tl2 > acc_cnn else 'CNN desde cero'
acc_mejor    = max(acc_tl2, acc_cnn)

y_pred = mejor_modelo.predict(X_te_mejor, batch_size=256, verbose=0).argmax(axis=1)

# Reporte
print(f"\nReporte del mejor modelo — {nombre_mejor}:")
print(classification_report(y_te, y_pred, target_names=NOMBRES_CLASES))

# Confusion matrix
cm = confusion_matrix(y_te, y_pred)
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cm, cmap='Blues')
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=8,
                color='white' if cm[i,j] > cm.max()*0.6 else 'black')
ax.set_xticks(range(10)); ax.set_xticklabels(NOMBRES_CLASES, rotation=45, ha='right')
ax.set_yticks(range(10)); ax.set_yticklabels(NOMBRES_CLASES)
ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
ax.set_title(f'{nombre_mejor} — Matriz de confusión (acc={acc_mejor:.4f})')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

In [ ]:
# Análisis de errores: ¿qué muestras fallan?
errores = np.where(y_pred != y_te)[0]
probas  = mejor_modelo.predict(X_te_mejor, batch_size=256, verbose=0)
confianza_errores = probas[errores].max(axis=1)

# Ordenar por alta confianza (errores más 'seguros' del modelo → peores)
orden = np.argsort(-confianza_errores)
peores = errores[orden[:16]]

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, idx in zip(axes.ravel(), peores):
    img = X_te_raw[idx]
    ax.imshow(img)
    clase_real  = NOMBRES_CLASES[y_te[idx]]
    clase_pred  = NOMBRES_CLASES[y_pred[idx]]
    conf        = probas[idx].max()
    ax.set_title(f'Real: {clase_real}\nPred: {clase_pred} ({conf:.2%})',
                 fontsize=8, color='red')
    ax.axis('off')
plt.suptitle('Peores errores: alta confianza en predicción incorrecta', fontsize=13)
plt.tight_layout(); plt.show()

# Estadísticas de errores
print(f"Total errores: {len(errores)} / {len(y_te)} ({len(errores)/len(y_te):.2%})")
print(f"Confianza media en errores:   {confianza_errores.mean():.4f}")
print(f"Confianza media en aciertos:  {probas[y_pred == y_te].max(axis=1).mean():.4f}")

In [ ]:
# Accuracy por clase + imágenes bien y mal clasificadas
print("Accuracy por clase:")
acc_por_clase = []
for c in range(10):
    mask = y_te == c
    acc_c = (y_pred[mask] == y_te[mask]).mean()
    acc_por_clase.append(acc_c)
    print(f"  {NOMBRES_CLASES[c]:10}: {acc_c:.4f} ({acc_c:.2%})")

fig, ax = plt.subplots(figsize=(11, 4))
colores = ['green' if a >= 0.85 else 'orange' if a >= 0.70 else 'red' for a in acc_por_clase]
bars = ax.bar(NOMBRES_CLASES, acc_por_clase, color=colores, edgecolor='white')
ax.axhline(acc_mejor, color='black', ls='--', lw=2, label=f'Accuracy global ({acc_mejor:.4f})')
ax.set_ylim(0, 1); ax.set_ylabel('Accuracy')
ax.set_title('Accuracy por clase'); ax.legend()
for bar, acc in zip(bars, acc_por_clase):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.2%}', ha='center', fontsize=9)
ax.grid(True, alpha=0.3, axis='y'); plt.tight_layout(); plt.show()

---
## Fase 6 — Guardar, cargar y usar en producción

In [ ]:
print("=" * 55)
print("FASE 6: GUARDAR Y DESPLEGAR")
print("=" * 55)

# Guardar el mejor modelo
mejor_modelo.save('cifar10_final.keras')
print("Modelo guardado: cifar10_final.keras")

# Guardar metadatos del proyecto
metadatos = {
    'proyecto':         'CIFAR-10 Clasificador de Imágenes',
    'dataset':          'CIFAR-10 (60,000 imágenes, 10 clases)',
    'clases':           NOMBRES_CLASES,
    'mejor_modelo':     nombre_mejor,
    'test_accuracy':    float(acc_mejor),
    'baseline_accuracy':float(acc_dense),
    'cnn_accuracy':     float(acc_cnn),
    'tl_accuracy':      float(acc_tl2),
    'input_shape':      [TARGET_SIZE, TARGET_SIZE, 3] if acc_tl2 > acc_cnn else [32, 32, 3],
    'preprocessing':    'normalizar a [0,1]'
}

with open('cifar10_metadatos.json', 'w', encoding='utf-8') as f:
    json.dump(metadatos, f, indent=2, ensure_ascii=False)
print("Metadatos guardados: cifar10_metadatos.json")

In [ ]:
# Función de predicción para producción
def predecir_imagen(imagen_array, modelo, clases, target_size=None, umbral_confianza=0.7):
    """
    Predice la clase de una imagen.

    Parámetros:
        imagen_array: np.ndarray (H, W, 3) con valores uint8 [0-255]
        modelo:       modelo Keras cargado
        clases:       lista de nombres de clase
        target_size:  (H, W) si el modelo necesita resize, None para mantener tamaño
        umbral_confianza: si la confianza es menor, reportar "incierto"

    Devuelve:
        dict con clase predicha, confianza y todas las probabilidades
    """
    # Preprocesar
    img = imagen_array.astype('float32') / 255.0
    if target_size:
        img = tf.image.resize(img, target_size).numpy()

    # Predecir
    probas = modelo.predict(img[None, ...], verbose=0)[0]  # (1, H, W, 3) → (n_clases,)
    clase_idx  = probas.argmax()
    confianza  = probas[clase_idx]

    return {
        'clase':     clases[clase_idx],
        'confianza': float(confianza),
        'incierto':  confianza < umbral_confianza,
        'top3': [
            (clases[i], float(probas[i]))
            for i in probas.argsort()[-3:][::-1]
        ]
    }

# Cargar el modelo desde disco (simulamos uso en producción)
modelo_cargado = keras.models.load_model('cifar10_final.keras')
print("Modelo cargado exitosamente")

ts = TARGET_SIZE if acc_tl2 > acc_cnn else None

# Probar con 8 imágenes
idx_muestra = np.random.choice(len(X_te_raw), 8, replace=False)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, i in zip(axes.ravel(), idx_muestra):
    resultado = predecir_imagen(X_te_raw[i], modelo_cargado, NOMBRES_CLASES, target_size=(ts,ts) if ts else None)
    clase_real = NOMBRES_CLASES[y_te[i]]
    ok = resultado['clase'] == clase_real
    ax.imshow(X_te_raw[i])
    titulo = f"Real: {clase_real}\nPred: {resultado['clase']} ({resultado['confianza']:.0%})"
    ax.set_title(titulo, color='green' if ok else 'red', fontsize=9)
    ax.axis('off')
plt.suptitle('Predicciones del modelo cargado desde disco', fontsize=13)
plt.tight_layout(); plt.show()

---
## Resumen del Proyecto

```
CIFAR-10 — Resumen de resultados:

  Baseline Dense:           ~50%  (sin estructura espacial)
  CNN desde cero:           ~80%  (arquitectura especializada)
  Transfer Learning (TL):   ~85%  (base de ImageNet)
  TL + Fine-tuning:         ~87%  (ajuste fino de capas superiores)

Técnicas aplicadas:
  ✓ Data Augmentation       → más variabilidad, menos overfitting
  ✓ BatchNormalization      → entrenamiento más rápido y estable
  ✓ Dropout                 → regularización
  ✓ L2 regularization       → pesos pequeños
  ✓ GlobalAveragePooling    → menos parámetros en clasificador
  ✓ CosineDecay schedule    → convergencia suave
  ✓ EarlyStopping           → parar en el momento óptimo
  ✓ ModelCheckpoint         → guardar el mejor modelo
  ✓ TensorBoard             → monitorización
  ✓ Transfer Learning       → aprovechar ImageNet (14M imágenes)
  ✓ Fine-tuning             → adaptar las capas superiores
  ✓ Análisis de errores     → entender las limitaciones
```

---
## Retos adicionales

### Reto 1 — Mejorar el modelo

¿Puedes superar el 90% en CIFAR-10?

Ideas:
- Usar EfficientNetB0 en lugar de MobileNetV2 (más preciso)
- Añadir más capas densas en el clasificador
- Probar ResNet50
- Mixup augmentation: mezclar imágenes de dos clases
- Label smoothing: `loss='categorical_crossentropy', label_smoothing=0.1`

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


---
### Reto 2 — Tu propio dataset

Aplica el mismo pipeline a un problema propio:

1. Recopila ≥200 imágenes de 2-3 categorías (plantas, objetos, animales...)
2. Organiza en carpetas: `datos/clase1/`, `datos/clase2/`
3. Carga con `keras.utils.image_dataset_from_directory`
4. Aplica Transfer Learning con MobileNetV2
5. Evalúa y analiza errores

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Ejemplo de carga desde directorio:
# dataset = keras.utils.image_dataset_from_directory(
#     'ruta/a/tus/datos',
#     image_size=(96, 96),
#     batch_size=32,
#     validation_split=0.2,
#     subset='training',
#     seed=42
# )

---
### Reto 3 — API de predicción

Crea una función `predecir_desde_url(url, modelo, clases)` que:

1. Descargue la imagen desde una URL
2. La preprocese automáticamente
3. Devuelva la predicción con confianza y top-3
4. Muestre la imagen con la predicción superpuesta

Pista: usa `requests`, `PIL.Image`, `io.BytesIO`

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


---
## Resumen del Módulo 8 — Deep Learning

```
C1  Historia y fundamentos    → perceptrón, redes multicapa, historia del DL
C2  Neuronas artificiales     → activaciones, inicialización, forward pass
C3  Backpropagation           → gradiente, optimizadores, Adam
C4  TensorFlow y Keras        → Sequential, compile, fit, evaluate
C5  Redes densas              → clasificación binaria/multiclase, diagnóstico
C6  Regularización            → Dropout, BatchNorm, L2, EarlyStopping
C7  CNN                       → Conv2D, Pooling, Transfer Learning
C8  RNN y Transformers        → LSTM, GRU, Embeddings, atención
C9  Entrenamiento             → TensorBoard, LR schedules, callbacks
C10 Proyecto                  → CIFAR-10 end-to-end: Dense→CNN→TL→FT
```

### ¿Qué viene en el Módulo 9?

En el próximo módulo aprenderemos a **desplegar modelos en producción**: APIs con FastAPI, contenedores Docker, monitorización en producción y las bases del MLOps.